# Tutorial 8: Analysing and validating an amorphous ensemble

Tutorials 2–6 make amorphous structures; this one asks whether an ensemble is right. It goes
through every descriptor `amorphgen --analyse` reports — partial RDFs, coordination, angles,
dimers, connectivity, rings, the X-ray structure factor S(q) and the total correlation
function T(r) — and ends
each section with the equivalent command line. Change the directory and the same cells
analyse any ensemble from Tutorials 2–6.

**Data**, shipped in `structures/`: two a-Ga₂O₃ ensembles of the AmorphGen software paper,
400 atoms, MACE-MPA-0.

| Directory | What | How |
|---|---|---|
| `structures/relaxed/` | 10 of the paper's 100 relaxed random cells | `amorphgen --random-gen --composition "Ga2O3*80" --target-density 4.84 --retry-mode none --relax -m mace-mpa-0 -C cubic` |
| `structures/annealed/` | 10 of the paper's 20 annealed cells | the same cells through `--hybrid-ensemble`: 50 ps at 3000 K, quench at 100 K/ps, 10 ps at 300 K (NVT), cubic-cell relaxation |

**Wall time** about one minute; the S(q) and T(r) transforms are the slow cells (~10 s each).
**Requires** the base package only (`pip install amorphgen`): no potential is used. Run from
this directory so that `structures/` resolves.

Unlike Tutorials 2–6, which use the teaching helpers in `Tutorials/analysis_helpers.py`,
this notebook calls the production `StructureAnalyser`, so every number is the one the
command line prints.

In [ ]:
# Run this notebook from its own tutorial directory (Tutorials/T8_validation_workflow/).
import matplotlib.pyplot as plt
import numpy as np

from amorphgen.analysis import StructureAnalyser

ensembles = {
    "relaxed": StructureAnalyser("structures/relaxed"),
    "annealed": StructureAnalyser("structures/annealed"),
}
colour = {"relaxed": "tab:blue", "annealed": "tab:red"}

for name, an in ensembles.items():
    atoms = an.atoms_list[0]
    L = atoms.cell.lengths()[0]
    print(f"{name:9s}: {len(an.atoms_list)} structures, {len(atoms)} atoms, "
          f"cell {L:.2f} Å, q_min = 2π/L = {2 * np.pi / L:.2f} Å⁻¹")

---
## 1. Short-range order: partial RDFs, coordination, angles

Cutoffs come from the first minimum of each partial RDF (`cutoff="auto-rdf"`, the CLI
default). Every mean is an equal-weight mean over structures with a Student-t interval, so
it carries the ensemble's uncertainty, not the pooled atoms'.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

pair_colour = {"Ga-O": "tab:blue", "Ga-Ga": "tab:orange", "O-O": "tab:green"}
for name, an in ensembles.items():
    ls = "-" if name == "relaxed" else "--"
    for pair, col in pair_colour.items():
        rdf = an.rdf(pair, rmax=6.0)
        axes[0].plot(rdf["r"], rdf["g_r"], ls, color=col, label=pair if name == "relaxed" else None)
    cn = an.coordination("Ga-O")["Ga-O"]["fraction_of_sites"]
    offset = -0.2 if name == "relaxed" else 0.2
    axes[1].bar([k + offset for k in cn], [100 * v for v in cn.values()], width=0.4,
                color=colour[name], edgecolor="black", label=name)
    ang = an.angle_distribution("Ga-O-Ga")["Ga-O-Ga"]
    axes[2].plot(ang["angle"], ang["distribution"], ls, color=colour[name], label=name)

axes[0].set(xlabel="r (Å)", ylabel="g(r)", xlim=(1.2, 6), title="Partial RDFs (solid relaxed, dashed annealed)")
axes[0].legend(frameon=False)
axes[1].set(xlabel="O neighbours of Ga", ylabel="fraction of Ga (%)", title="Ga–O coordination")
axes[1].legend(frameon=False)
axes[2].set(xlabel="Ga–O–Ga angle (°)", ylabel="probability density", xlim=(80, 180), title="Bridging angle")
axes[2].legend(frameon=False)
fig.tight_layout()

for name, an in ensembles.items():
    b = an.bond_distances("Ga-O")["Ga-O"]
    c = an.coordination("Ga-O")
    a = an.bond_angles("Ga-O-Ga")["Ga-O-Ga"]
    d = an.density()
    u = lambda x: f"{x['uncertainty']['mean']:.3f} [{x['uncertainty']['ci_low']:.3f}, {x['uncertainty']['ci_high']:.3f}]"
    print(f"{name:9s}  density {u(d)} g/cm³   Ga–O {u(b)} Å   "
          f"CN Ga–O {u(c['Ga-O'])}   O–Ga {u(c['O-Ga'])}   Ga–O–Ga {a['uncertainty']['mean']:.1f}°")

Both ensembles share the same short-range order: Ga–O 1.93 Å, 4.5 O around each Ga (56 %
four-fold, 40 % five-fold), and a Ga–O–Ga distribution with maxima near 95° and 120°. The
95° peak comes from edge-sharing pairs (two Ga sharing two O), the 120° peak from
corner-sharing ones; section 2 counts them, and the anneal moves weight from the first to
the second. These are the paper's values for the full ensembles (1.93 Å, CN 4.5).

```bash
amorphgen --analyse --input-dir structures/relaxed --save-plot analysis_relaxed
```
writes `analysis_rdf`, `analysis_cn`, `analysis_angles` and `analysis_density` (`.png/.csv`)
and the per-structure tables; `--smearing` sets the Gaussian width of the plotted g(r).

---
## 2. Beyond the averages: dimers, connectivity, rings

- **O–O dimers** — contacts below 0.85 × the radii-derived minimum separation, left when a
  cold relaxation collapses an under-coordinated seed (`dimer_report()`).
- **Polyhedral connectivity** — two Ga sharing one O are corner-sharing, two O
  edge-sharing, three or more face-sharing; reported with the fraction of Ga in at least
  one edge/face-sharing pair (`polyhedral_connectivity()`).
- **Rings** of the Ga–O network — the shortest ring through each edge (`ring_statistics()`).
  In a close-packed oxide nearly every edge closes a 3-ring, so the statistic is
  uninformative here; it is the one that separates a good from a poor model of a network
  glass (SiO₂, GeO₂), where an anneal builds the 5–7-membered rings.

In [ ]:
def defect_summary(an, label):
    d = an.dimer_report()
    c = an.polyhedral_connectivity()
    r = an.ring_statistics()
    lp = c["link_percent"]
    print(f"{label} (n = {len(an.atoms_list)})")
    print(f"  O–O dimers: {d['total']} in total; {100 * d['fraction_of_structures']:.0f}% of structures "
          f"have at least one; {100 * d['fraction_of_sites']:.1f}% of sites")
    print(f"  Ga–Ga polyhedral links: corner {lp['corner']:.1f}%, edge {lp['edge']:.1f}%, face {lp['face']:.1f}%")
    print(f"  Ga in at least one edge/face-sharing pair: {c['cation_edge_or_face_percent']:.1f}%")
    print("  ring sizes (fraction of network edges): "
          + ", ".join(f"{s}: {f:.1f}%" for s, f in zip(r["ring_sizes"], r["fractions"]) if f > 0.05))

for name, an in ensembles.items():
    defect_summary(an, name)

This is what the averages hide: six of the ten relaxed cells contain an O–O dimer (0.4 % of
the atoms) and 74 % of the Ga are in an edge-sharing pair; after the anneal there are no
dimers and 67 % are (the paper: 58 % and 74 % → 0 and 69 %). The ring statistics are the
same in both, 99 % 3-rings, as a close-packed oxide gives.

```bash
amorphgen --analyse --input-dir structures/relaxed --check-dimers --connectivity --rings
```
For several cations (a-InGaZnO₄) every cation is a ring node; `--rings In/Ga/Zn-O` names
the node group explicitly.

---
## 3. The X-ray structure factor S(q)

`structure_factor_direct()` computes the Faber–Ziman S(q) from the scattering sums over
the reciprocal lattice of each cell, which keeps the low-q intensities that a Fourier
transform of a truncated g(r) loses. Two choices decide whether it can be set against a
measurement:

- **Weighting** — X-ray (atomic form factors) or neutron (scattering lengths); the curves
  differ, so use the experiment's (`weighting="xray"` / `"neutron"`).
- **Box size** — the lowest q is 2π/L: 0.36 Å⁻¹ for these 17.4 Å cells, so the 1–2 Å⁻¹ region
  is well sampled; an 80-atom cell samples it with a handful of q-vectors and the analyser
  warns. `sigma_q` re-bins the discrete shells; the raw shells stay in `s_q_raw`.

A measured curve goes in as a two-column file with `--experiment-sq FILE`; the report then
states the agreement over `--sq-fit-range` against the ensemble's confidence band. (The
paper compared with a sputtered film, Liu et al., *Adv. Mater.* 35, 2210873, 2023; that
curve is not redistributed here.)

In [ ]:
sq = {name: an.structure_factor_direct(qmax=12.0, nq=240, weighting="xray", sigma_q=0.05)
      for name, an in ensembles.items()}
sq_neutron = ensembles["relaxed"].structure_factor_direct(qmax=12.0, nq=240,
                                                          weighting="neutron", sigma_q=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, s in sq.items():
    q, y = np.asarray(s["q"], float), np.asarray(s["s_q"], float)
    axes[0].plot(q, y, color=colour[name], label=f"{name} (X-ray)")
    lo, hi = np.asarray(s["uncertainty"]["ci_low"], float), np.asarray(s["uncertainty"]["ci_high"], float)
    axes[0].fill_between(q, lo, hi, color=colour[name], alpha=0.2)
axes[0].axvspan(1.0, 2.0, color="0.9", zorder=0)
axes[0].set(xlabel="q (Å⁻¹)", ylabel="S(q)", xlim=(0, 12), title="X-ray S(q), 95 % band; grey: where an FSDP would sit")
axes[0].legend(frameon=False)

q = np.asarray(sq["relaxed"]["q"], float)
axes[1].plot(q, np.asarray(sq["relaxed"]["s_q"], float), color="tab:blue", label="X-ray weighting")
axes[1].plot(q, np.asarray(sq_neutron["s_q"], float), color="tab:green", label="neutron weighting")
axes[1].set(xlabel="q (Å⁻¹)", ylabel="S(q)", xlim=(0, 12), title="Same relaxed cells, two weightings")
axes[1].legend(frameon=False)
fig.tight_layout()

q_finite = q[np.isfinite(np.asarray(sq["relaxed"]["s_q"], float))]
print(f"lowest resolved q: {q_finite[0]:.2f} Å⁻¹  (2π/L)")
for name, s in sq.items():
    y = np.asarray(s["s_q"], float)
    sel = (q > 1.5) & (q < 3.5)
    print(f"{name:9s}: first peak at q = {q[sel][np.nanargmax(y[sel])]:.2f} Å⁻¹, S = {np.nanmax(y[sel]):.2f}")

The two X-ray curves agree within their 95 % bands from the first peak at 2.4 Å⁻¹ upward;
the anneal sharpens that peak (S = 1.59 → 1.73) and creates nothing below it, because
a-Ga₂O₃ has no first sharp diffraction peak (the grey band marks where one would sit).
Under neutron weighting the 2.4 Å⁻¹ peak becomes a shoulder: Ga–Ga carries 52 % of the
X-ray signal but 21 % of the neutron signal, O–O 8 % and 30 %. Compare under the weighting
of the measurement.

---
## 4. The total correlation function T(r)

Diffraction papers plot T(r) = 4πrρ g(r), with g(r) from the Fourier transform of the
*weighted* S(q) over the *measured* q range, usually with a Lorch window.
`total_correlation()` does the same to the computed S(q), so the curve is comparable with a
published one — unlike the unweighted total g(r) of section 1. It also returns the reduced
PDF G(r) = 4πrρ[g(r) − 1]. Set `qmin`, `qmax` and `window` to the experiment's values;
`--tr-scan` sweeps them and reports how far the first peak and its integral move, the
transform's own uncertainty.

In [ ]:
tr = {name: an.total_correlation(weighting="xray", qmin=0.4, qmax=12.0, window="lorch")
      for name, an in ensembles.items()}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, t in tr.items():
    r = np.asarray(t["r"], float)
    axes[0].plot(r, np.asarray(t["T_r"], float), color=colour[name], label=name)
    axes[1].plot(r, np.asarray(t["G_r"], float), color=colour[name], label=name)
axes[0].set(xlabel="r (Å)", ylabel="T(r) (Å⁻²)", xlim=(1, 8), title="T(r), X-ray weighting, q ≤ 12 Å⁻¹, Lorch")
axes[1].set(xlabel="r (Å)", ylabel="G(r) (Å⁻²)", xlim=(1, 8), title="Reduced PDF G(r)")
axes[1].axhline(0, color="0.7", lw=0.8)
for ax in axes:
    ax.legend(frameon=False)
fig.tight_layout()

for name, t in tr.items():
    r, T = np.asarray(t["r"], float), np.asarray(t["T_r"], float)
    sel = (r > 1.5) & (r < 2.5)
    print(f"{name:9s}: first T(r) peak at r = {r[sel][np.argmax(T[sel])]:.3f} Å  "
          f"(Ga–O bond length from section 1: "
          f"{ensembles[name].bond_distances('Ga-O')['Ga-O']['uncertainty']['mean']:.3f} Å)")

The first T(r) peak at 1.92 Å is the Ga–O shell, 0.01 Å below the mean bond length of
section 1 because the finite-q transform with a Lorch window broadens and shifts it — the
offset `--tr-scan` quantifies and a measured curve carries too. The second peak at 3.3 Å is
Ga–Ga, sharpened by the anneal as the short Ga–Ga distances of edge-sharing pairs go. The
integral over the first peak is a scattering-weighted count, not the Ga–O coordination of
section 1: coordination numbers come from the neighbour count, T(r) is for diffraction.

```bash
amorphgen --analyse --input-dir structures/relaxed \
    --sq --sq-weighting xray --sq-smooth 0.05 --sq-qmax 12 \
    --tr --tr-qrange 0.4 12 --tr-window lorch --tr-scan \
    --save-plot analysis_relaxed
# with a measured curve:  --experiment-sq film_sq.txt --sq-fit-range 1 7
```
writes `analysis_sq` and `analysis_tr` (`.png/.csv`, with the per-structure and
uncertainty tables).

---
## 5. Relaxed versus annealed

The annealed cells are the relaxed ones after stages 4–7 at fixed cell
(`--hybrid-ensemble`). Side by side, with the potential energy per atom that the structure
files carry in their headers (`energy_ranking()` reads it):

In [ ]:
def row(name):
    an = ensembles[name]
    c = an.polyhedral_connectivity()
    d = an.dimer_report()
    energies = an.energy_ranking()["energies_per_atom"]
    e = np.mean(list(energies.values()) if isinstance(energies, dict) else energies)
    return [an.density()["mean"], an.bond_distances("Ga-O")["Ga-O"]["mean"],
            an.coordination("Ga-O")["Ga-O"]["mean"], an.bond_angles("Ga-O-Ga")["Ga-O-Ga"]["mean"],
            c["cation_edge_or_face_percent"], 100 * d["fraction_of_structures"], e]

names = ["density (g/cm³)", "Ga–O (Å)", "CN Ga–O", "Ga–O–Ga (°)",
         "Ga in edge/face pairs (%)", "structures with dimers (%)", "energy (eV/atom)"]
before, after = row("relaxed"), row("annealed")
print(f"{'':28s}{'relaxed':>10s}{'annealed':>10s}")
for label, b, a in zip(names, before, after):
    print(f"{label:28s}{b:10.3f}{a:10.3f}")
print(f"\nThe anneal lowers the energy by {1000 * (before[-1] - after[-1]):.0f} meV/atom.")

The anneal lowers the energy by about 52 meV/atom (the paper: 51) and changes the defect
population, not the structure: the dimers go, edge sharing drops six points, the density
settles 0.08 g/cm³ higher, and bond length, coordination and angle move by less than the
width of their reference ranges. For an oxide only ever made as a film, this is the paper's
reading: the relaxed random ensemble models the as-deposited film and the anneal is the
annealing step. A glass former gives the opposite table — in a-SiO₂ the same anneal lowers
the energy by 130–290 meV/atom, makes the 17 % non-fourfold Si fourfold, and builds the
first sharp diffraction peak no relaxed random cell has (`examples/softwarex_run_sio2_*.slurm`).

---
## Going further

- **Your own ensemble**: point `StructureAnalyser` or `--analyse --input-dir` at the
  `random_opt/` of Tutorial 2 or 3 or the `final/` of Tutorial 5.
- **Everything at once**, the paper's `examples/softwarex_analyse_oxide.slurm`:
  ```bash
  amorphgen --analyse --input-dir DIR --rings --connectivity --check-dimers \
      --sq --sq-weighting xray --sq-smooth 0.05 --tr \
      --save-plot analysis --save-report analysis/report.txt
  ```
- **The full ensembles**: `examples/softwarex_run_oxide.slurm` regenerates the 100 relaxed
  and 20 annealed cells on a GPU node (see the HPC guide).
- **Other descriptors** from the same analyser: `voronoi()`, `void_distribution()`,
  `oxygen_speciation()`, `xrd_pattern()`, `elastic_moduli()` and `vibrational_dos()` (these
  two need a calculator), `compare_experiment()` for a measured S(q) or T(r); on the
  command line, `--reference FILE` scores the ensemble against literature ranges and
  `--convergence` estimates how many structures a tolerance needs.